In [1]:
"""
================================================================================
Experiment 35: MFCC + CNN (2D) + XGBoost + Class Weights (Updated Output Format)
================================================================================
This script performs patient-level 3-fold cross-validation for:
- Feature extraction: MFCC only (40 coefficients, full time-frequency representation)
- Method: CNN (2D) features + XGBoost classifier with class weights
- Minor classes (Healthy, URTI) are merged into 'Other' for stable stratification.
- Calculates per-class metrics and final summary with old naming convention.

Author: Ehab Hassan Abdel Hady
Date: 2025
"""

import os
import numpy as np
import pandas as pd
import librosa
import soundfile as sf
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, f1_score, recall_score,
    balanced_accuracy_score, confusion_matrix,
    precision_recall_fscore_support
)
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dropout, GlobalAveragePooling2D, Input
import xgboost as xgb
from sklearn.utils.class_weight import compute_class_weight
import warnings
warnings.filterwarnings('ignore')
from collections import Counter
import json
import pickle
import time
from datetime import timedelta
from joblib import Parallel, delayed
from tqdm import tqdm
import gc

import tensorflow as tf
tf.get_logger().setLevel('ERROR')

# =============================================================================
# Configuration – EDIT THESE PATHS AND OPTIONS
# =============================================================================
SOUND_FILES_PATH = "E:\\FFF\\respiratory-sound-database\\Respiratory_Sound_Database\\Respiratory_Sound_Database\\audio_and_txt_files\\"
DIAGNOSIS_CSV_PATH = "E:\\FFF\\respiratory-sound-database\\Respiratory_Sound_Database\\Respiratory_Sound_Database\\patient_diagnosis.csv"
OUTPUT_DIR = "E:\\Results\\exp35_updated\\"          # change for each experiment

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Checkpoint file paths (same as before)
FEATURES_RAW_FILE = os.path.join(OUTPUT_DIR, "features_raw.npy")
LABELS_RAW_FILE = os.path.join(OUTPUT_DIR, "labels_raw.npy")
PATIENT_IDS_RAW_FILE = os.path.join(OUTPUT_DIR, "patient_ids_raw.npy")
CLEANED_DATA_FILE = os.path.join(OUTPUT_DIR, "cleaned_data.pkl")
PROGRESS_FILE = os.path.join(OUTPUT_DIR, "progress.json")
RESULTS_FILE = os.path.join(OUTPUT_DIR, "fold_results.csv")
PER_CLASS_RESULTS_FILE = os.path.join(OUTPUT_DIR, "per_class_results.csv")
TIMING_FILE = os.path.join(OUTPUT_DIR, "timing_log.txt")
FINAL_SUMMARY_FILE = os.path.join(OUTPUT_DIR, "final_summary.csv")

# Number of parallel jobs for feature extraction
N_JOBS = 8
MAX_FILES = None  # set to None to process all files
USE_PARALLEL = False  # serial for stability
if USE_PARALLEL:
    N_JOBS = 8
else:
    N_JOBS = 1

DTYPE = np.float32
CNN_BATCH_SIZE = 32

# =============================================================================
# Utility functions
# =============================================================================
def save_progress(stage, fold=None):
    progress = {'stage': stage}
    if fold is not None:
        progress['last_completed_fold'] = fold
    with open(PROGRESS_FILE, 'w') as f:
        json.dump(progress, f)

def load_progress():
    if os.path.exists(PROGRESS_FILE):
        with open(PROGRESS_FILE, 'r') as f:
            progress = json.load(f)
        return progress.get('stage', 1), progress.get('last_completed_fold', 0)
    return 1, 0

def log_time(stage_name, elapsed_seconds):
    line = f"{stage_name}: {timedelta(seconds=elapsed_seconds)} (hh:mm:ss)\n"
    with open(TIMING_FILE, 'a') as f:
        f.write(line)
    print(line.strip())

# =============================================================================
# Feature extraction (MFCC only, 2D)
# =============================================================================
def extract_features_single(file_path, max_pad_len=862):
    try:
        audio, sr = sf.read(file_path)
        if len(audio.shape) > 1:
            audio = np.mean(audio, axis=1)
        target_len = 20 * sr
        if len(audio) > target_len:
            audio = audio[:target_len]
        elif len(audio) < target_len:
            repeats = target_len // len(audio) + 1
            audio = np.tile(audio, repeats)[:target_len]

        mfccs = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=40).astype(DTYPE)

        if mfccs.shape[1] < max_pad_len:
            pad_width = max_pad_len - mfccs.shape[1]
            mfccs = np.pad(mfccs, pad_width=((0, 0), (0, pad_width)), mode='constant')
        else:
            mfccs = mfccs[:, :max_pad_len]

        return mfccs.astype(DTYPE)
    except Exception as e:
        print(f"\n!!! Error processing {file_path}: {e}")
        return None

# =============================================================================
# Per-class metrics
# =============================================================================
def per_class_metrics(y_true, y_pred, class_names):
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=range(len(class_names)), zero_division=0
    )
    cm = confusion_matrix(y_true, y_pred, labels=range(len(class_names)))
    FP = cm.sum(axis=0) - np.diag(cm)
    FN = cm.sum(axis=1) - np.diag(cm)
    TP = np.diag(cm)
    TN = cm.sum() - (FP + FN + TP)

    with np.errstate(divide='ignore', invalid='ignore'):
        FPR = np.divide(FP, FP + TN, where=(FP+TN) > 0)
        FNR = np.divide(FN, FN + TP, where=(FN+TP) > 0)
        FPR = np.nan_to_num(FPR, nan=0.0)
        FNR = np.nan_to_num(FNR, nan=0.0)

    result = {}
    for i, name in enumerate(class_names):
        result[name] = {
            'precision': precision[i],
            'recall': recall[i],
            'f1': f1[i],
            'fpr': FPR[i],
            'fnr': FNR[i]
        }
    return result

# =============================================================================
# Main execution
# =============================================================================
print("=" * 60)
print("Experiment 35: MFCC + CNN (2D) + XGBoost + Class Weights (Updated Output)")
print("=" * 60)

current_stage, last_fold = load_progress()
print(f"Resuming from stage {current_stage}, last completed fold: {last_fold}")

# ------------------ STAGE 1: Extract raw features ------------------
if current_stage == 1:
    stage_start = time.time()
    print("\n[Stage 1] Extracting MFCC features from audio files...")

    all_filenames = [f for f in os.listdir(SOUND_FILES_PATH)
                     if f.endswith('.wav') and os.path.isfile(os.path.join(SOUND_FILES_PATH, f))]
    if MAX_FILES is not None:
        filenames = all_filenames[:MAX_FILES]
        print(f"    Using first {len(filenames)} files (out of {len(all_filenames)} total).")
    else:
        filenames = all_filenames
        print(f"    Found {len(filenames)} audio files.")

    filepaths = [os.path.join(SOUND_FILES_PATH, f) for f in filenames]
    patient_ids = [int(name[:3]) for name in filenames]

    p_diag = pd.read_csv(DIAGNOSIS_CSV_PATH, header=None)
    diagnosis_dict = dict(zip(p_diag[0], p_diag[1]))
    labels = [diagnosis_dict.get(pid, None) for pid in patient_ids]

    if USE_PARALLEL:
        print(f"    Using {N_JOBS} parallel workers...")
        results = Parallel(n_jobs=N_JOBS, verbose=10)(
            delayed(extract_features_single)(fp) for fp in tqdm(filepaths, desc="Feature extraction")
        )
    else:
        print(f"    Using serial processing...")
        results = []
        for fp in tqdm(filepaths, desc="Feature extraction"):
            results.append(extract_features_single(fp))
            gc.collect()

    features = []
    valid_indices = []
    error_files = []
    for i, feat in enumerate(results):
        if feat is not None:
            features.append(feat)
            valid_indices.append(i)
        else:
            error_files.append(filepaths[i])

    if error_files:
        print(f"    Errors occurred on {len(error_files)} files. First few: {error_files[:5]}")
    print(f"    Extraction complete: {len(features)} successful, {len(error_files)} errors.")
    if len(features) == 0:
        raise RuntimeError("No features extracted. Check error messages above.")

    features = np.array(features, dtype=DTYPE)  # shape: (n_samples, 40, 862)
    patient_ids = [patient_ids[i] for i in valid_indices]
    labels = [labels[i] for i in valid_indices]

    np.save(FEATURES_RAW_FILE, features)
    np.save(LABELS_RAW_FILE, labels)
    np.save(PATIENT_IDS_RAW_FILE, patient_ids)
    print(f"    Raw features saved to {FEATURES_RAW_FILE}")

    elapsed = time.time() - stage_start
    log_time("Stage 1 - Feature extraction", elapsed)
    save_progress(2)
    current_stage = 2

# ------------------ STAGE 2: Clean and prepare data ------------------
if current_stage == 2:
    stage_start = time.time()
    print("\n[Stage 2] Cleaning and preparing data...")

    features = np.load(FEATURES_RAW_FILE, allow_pickle=True).astype(DTYPE)
    labels = np.load(LABELS_RAW_FILE, allow_pickle=True).tolist()
    patient_ids = np.load(PATIENT_IDS_RAW_FILE, allow_pickle=True).tolist()

    valid = [i for i, lbl in enumerate(labels) if lbl is not None]
    features = features[valid]
    patient_ids = [patient_ids[i] for i in valid]
    labels = [labels[i] for i in valid]

    new_features = []
    new_labels = []
    new_patient_ids = []
    for f, lbl, pid in zip(features, labels, patient_ids):
        if lbl in ['Asthma', 'LRTI']:
            continue
        elif lbl in ['Healthy', 'URTI']:
            new_labels.append('Other')
            new_features.append(f)
            new_patient_ids.append(pid)
        else:
            new_labels.append(lbl)
            new_features.append(f)
            new_patient_ids.append(pid)

    features = np.array(new_features, dtype=DTYPE)
    labels = new_labels
    patient_ids = new_patient_ids

    print(f"    Remaining samples after cleaning: {len(features)}")
    print("    New class distribution:")
    print(Counter(labels))

    # Add channel dimension for CNN (2D expects (height, width, channels))
    features = features[..., np.newaxis]  # shape: (n_samples, 40, 862, 1)

    cleaned_data = {
        'features': features,
        'labels': labels,
        'patient_ids': patient_ids
    }
    with open(CLEANED_DATA_FILE, 'wb') as f:
        pickle.dump(cleaned_data, f)
    print(f"    Cleaned data saved to {CLEANED_DATA_FILE}")

    elapsed = time.time() - stage_start
    log_time("Stage 2 - Data cleaning", elapsed)
    save_progress(3)
    current_stage = 3

# ------------------ STAGE 3: Cross-validation ------------------
if current_stage == 3:
    stage_start = time.time()
    print("\n[Stage 3] Running cross‑validation...")

    with open(CLEANED_DATA_FILE, 'rb') as f:
        cleaned = pickle.load(f)
    features = cleaned['features'].astype(DTYPE)
    labels = cleaned['labels']
    patient_ids = cleaned['patient_ids']

    df = pd.DataFrame({
        'features': list(features),
        'label': labels,
        'patient_id': patient_ids
    })

    unique_patients = df.groupby('patient_id')['label'].first().reset_index()
    le = LabelEncoder()
    unique_patients['label_enc'] = le.fit_transform(unique_patients['label'])

    print(f"    Unique patients: {len(unique_patients)}")
    print(f"    Class distribution among patients:\n{unique_patients['label'].value_counts()}")

    n_splits = 3
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    if os.path.exists(RESULTS_FILE):
        fold_results_df = pd.read_csv(RESULTS_FILE)
        fold_results = fold_results_df.to_dict('records')
        completed_folds = set(fold_results_df['fold'].tolist())
    else:
        fold_results = []
        completed_folds = set()

    if os.path.exists(PER_CLASS_RESULTS_FILE):
        per_class_df = pd.read_csv(PER_CLASS_RESULTS_FILE)
        per_class_fold_data = per_class_df.to_dict('records')
    else:
        per_class_fold_data = []

    for fold, (train_pat_idx, test_pat_idx) in enumerate(skf.split(
            unique_patients['patient_id'], unique_patients['label_enc'])):
        fold_num = fold + 1
        if fold_num in completed_folds:
            print(f"\n--- Fold {fold_num} already completed, skipping ---")
            continue

        fold_start = time.time()
        print(f"\n--- Fold {fold_num}/{n_splits} ---")

        train_patients = unique_patients.iloc[train_pat_idx]['patient_id'].values
        test_patients = unique_patients.iloc[test_pat_idx]['patient_id'].values

        train_df = df[df['patient_id'].isin(train_patients)]
        test_df = df[df['patient_id'].isin(test_patients)]

        X_train = np.stack(train_df['features'].values).astype(DTYPE)
        X_test = np.stack(test_df['features'].values).astype(DTYPE)
        y_train_str = train_df['label'].values
        y_test_str = test_df['label'].values

        le_fold = LabelEncoder()
        y_train = le_fold.fit_transform(y_train_str)
        y_test = le_fold.transform(y_test_str)
        classes = np.unique(y_train)

        print(f"    Train samples: {len(X_train)}, Test samples: {len(X_test)}")
        print(f"    Train patients: {len(train_patients)}, Test patients: {len(test_patients)}")
        print(f"    Classes in train: {le_fold.inverse_transform(classes)}")

        # --- CNN feature extractor (same architecture) ---
        cnn = Sequential([
            Input(shape=(40, 862, 1)),
            Conv2D(16, kernel_size=2, activation='relu'),
            MaxPooling2D(pool_size=2),
            Dropout(0.2),
            Conv2D(32, kernel_size=2, activation='relu'),
            MaxPooling2D(pool_size=2),
            Dropout(0.2),
            Conv2D(64, kernel_size=2, activation='relu'),
            MaxPooling2D(pool_size=2),
            Dropout(0.2),
            Conv2D(128, kernel_size=2, activation='relu'),
            MaxPooling2D(pool_size=2),
            Dropout(0.2),
            GlobalAveragePooling2D()
        ])
        cnn.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

        X_train_feat = cnn.predict(X_train, verbose=0, batch_size=CNN_BATCH_SIZE)
        X_test_feat = cnn.predict(X_test, verbose=0, batch_size=CNN_BATCH_SIZE)
        X_train_feat = X_train_feat.reshape(X_train_feat.shape[0], -1).astype(DTYPE)
        X_test_feat = X_test_feat.reshape(X_test_feat.shape[0], -1).astype(DTYPE)

        # Class weights
        class_weights = compute_class_weight('balanced', classes=classes, y=y_train)
        sample_weights = np.array([class_weights[y] for y in y_train])

        xgb_model = xgb.XGBClassifier(
            n_estimators=200,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.9,
            colsample_bytree=0.8,
            gamma=0.1,
            use_label_encoder=False,
            eval_metric='mlogloss',
            random_state=42
        )
        xgb_model.fit(X_train_feat, y_train, sample_weight=sample_weights)
        y_pred = xgb_model.predict(X_test_feat)

        # Metrics calculation
        cm = confusion_matrix(y_test, y_pred, labels=classes)
        FP = cm.sum(axis=0) - np.diag(cm)
        FN = cm.sum(axis=1) - np.diag(cm)
        TP = np.diag(cm)
        TN = cm.sum() - (FP + FN + TP)

        with np.errstate(divide='ignore', invalid='ignore'):
            FPR_per_class = np.divide(FP, FP + TN, where=(FP+TN) > 0)
            FNR_per_class = np.divide(FN, FN + TP, where=(FN+TP) > 0)
            FPR_per_class = np.nan_to_num(FPR_per_class, nan=0.0)
            FNR_per_class = np.nan_to_num(FNR_per_class, nan=0.0)

        macro_fpr = np.mean(FPR_per_class)
        macro_fnr = np.mean(FNR_per_class)

        fold_metrics = {
            'fold': fold_num,
            'accuracy': accuracy_score(y_test, y_pred),
            'macro_f1': f1_score(y_test, y_pred, average='macro', zero_division=0),
            'macro_recall': recall_score(y_test, y_pred, average='macro', zero_division=0),
            'weighted_f1': f1_score(y_test, y_pred, average='weighted', zero_division=0),
            'weighted_recall': recall_score(y_test, y_pred, average='weighted', zero_division=0),
            'balanced_accuracy': balanced_accuracy_score(y_test, y_pred),
            'fpr': macro_fpr,
            'fnr': macro_fnr
        }
        fold_results.append(fold_metrics)

        class_names = le_fold.classes_
        per_class = per_class_metrics(y_test, y_pred, class_names)
        for class_name, metrics in per_class.items():
            per_class_fold_data.append({
                'fold': fold_num,
                'class': class_name,
                'precision': metrics['precision'],
                'recall': metrics['recall'],
                'f1': metrics['f1'],
                'fpr': metrics['fpr'],
                'fnr': metrics['fnr']
            })

        pd.DataFrame(fold_results).to_csv(RESULTS_FILE, index=False)
        pd.DataFrame(per_class_fold_data).to_csv(PER_CLASS_RESULTS_FILE, index=False)
        save_progress(3, fold_num)

        fold_elapsed = time.time() - fold_start
        print(f"    Fold {fold_num} completed in {timedelta(seconds=fold_elapsed)}.")
        print(f"    Results saved.")

    elapsed = time.time() - stage_start
    log_time("Stage 3 - Cross-validation", elapsed)
    print("\nAll folds completed.")
    save_progress(4)

# ------------------ STAGE 4: Aggregate final results with old naming ------------------
if current_stage == 4 or (current_stage >= 3 and os.path.exists(RESULTS_FILE)):
    print("\n[Stage 4] Aggregating final results (old naming)...")
    fold_results_df = pd.read_csv(RESULTS_FILE)
    fold_results = fold_results_df.to_dict('records')

    # Calculate mean and std for each metric
    acc_mean = np.mean([f['accuracy'] for f in fold_results])
    acc_std = np.std([f['accuracy'] for f in fold_results])
    macro_f1_mean = np.mean([f['macro_f1'] for f in fold_results])
    macro_f1_std = np.std([f['macro_f1'] for f in fold_results])
    macro_recall_mean = np.mean([f['macro_recall'] for f in fold_results])
    macro_recall_std = np.std([f['macro_recall'] for f in fold_results])
    weighted_f1_mean = np.mean([f['weighted_f1'] for f in fold_results])
    weighted_f1_std = np.std([f['weighted_f1'] for f in fold_results])
    weighted_recall_mean = np.mean([f['weighted_recall'] for f in fold_results])
    weighted_recall_std = np.std([f['weighted_recall'] for f in fold_results])
    balanced_acc_mean = np.mean([f['balanced_accuracy'] for f in fold_results])
    balanced_acc_std = np.std([f['balanced_accuracy'] for f in fold_results])
    fpr_mean = np.mean([f['fpr'] for f in fold_results])
    fpr_std = np.std([f['fpr'] for f in fold_results])
    fnr_mean = np.mean([f['fnr'] for f in fold_results])
    fnr_std = np.std([f['fnr'] for f in fold_results])

    # Build DataFrame with old naming
    summary = {
        'Balanced Accuracy (mean)': [balanced_acc_mean],
        'Balanced Accuracy (std)': [balanced_acc_std],
        'Macro F1-Score (mean)': [macro_f1_mean],
        'Macro F1-Score (std)': [macro_f1_std],
        'Weighted-F1 Score (mean)': [weighted_f1_mean],
        'Weighted-F1 Score (std)': [weighted_f1_std],
        'Macro-Recall (mean)': [macro_recall_mean],
        'Macro-Recall (std)': [macro_recall_std],
        'Weighted-Recall (mean)': [weighted_recall_mean],
        'Weighted-Recall (std)': [weighted_recall_std],
        'Standard Accuracy (mean)': [acc_mean],
        'Standard Accuracy (std)': [acc_std],
        'Macro False Positive Rate (mean)': [fpr_mean],
        'Macro False Positive Rate (std)': [fpr_std],
        'Macro False Negative Rate (mean)': [fnr_mean],
        'Macro False Negative Rate (std)': [fnr_std]
    }
    summary_df = pd.DataFrame(summary)

    print("\n" + "=" * 50)
    print("FINAL SUMMARY (old naming)")
    print("=" * 50)
    print(summary_df.to_string(index=False))

    summary_df.to_csv(FINAL_SUMMARY_FILE, index=False)
    print(f"\nFinal summary saved to {FINAL_SUMMARY_FILE}")

    if os.path.exists(TIMING_FILE):
        print("\n" + "=" * 50)
        print("TIMING LOG")
        with open(TIMING_FILE, 'r') as f:
            print(f.read())
    else:
        print("\nNo timing log found.")

print("\n" + "=" * 60)
print("Experiment 35 completed successfully.")
print("=" * 60)

Experiment 35: MFCC + CNN (2D) + XGBoost + Class Weights (Updated Output)
Resuming from stage 1, last completed fold: 0

[Stage 1] Extracting MFCC features from audio files...
    Found 920 audio files.
    Using serial processing...


Feature extraction: 100%|██████████| 920/920 [03:48<00:00,  4.02it/s]


    Extraction complete: 920 successful, 0 errors.
    Raw features saved to E:\Results\exp35_updated\features_raw.npy
Stage 1 - Feature extraction: 0:03:49.185936 (hh:mm:ss)

[Stage 2] Cleaning and preparing data...
    Remaining samples after cleaning: 917
    New class distribution:
Counter({'COPD': 793, 'Other': 58, 'Pneumonia': 37, 'Bronchiectasis': 16, 'Bronchiolitis': 13})
    Cleaned data saved to E:\Results\exp35_updated\cleaned_data.pkl
Stage 2 - Data cleaning: 0:00:00.485352 (hh:mm:ss)

[Stage 3] Running cross‑validation...
    Unique patients: 123
    Class distribution among patients:
label
COPD              64
Other             40
Bronchiectasis     7
Pneumonia          6
Bronchiolitis      6
Name: count, dtype: int64

--- Fold 1/3 ---
    Train samples: 655, Test samples: 262
    Train patients: 82, Test patients: 41
    Classes in train: ['Bronchiectasis' 'Bronchiolitis' 'COPD' 'Other' 'Pneumonia']
    Fold 1 completed in 0:00:03.316477.
    Results saved.

--- Fold 2/3